# Generative Adversarial Network (GAN) - TensorFlow / Keras

**Goal:** Train a small generator/discriminator pair on digits.

This notebook favors clear, production-style structure: seeded runs,
explicit data preparation, small reusable modules, and compact
training loops that can be expanded for larger experiments.


## Architecture Notes

- **What it learns:** A generator and discriminator improve through adversarial feedback.
- **Where it is used:** image synthesis, augmentation research, and generator-discriminator demos.
- **Why it works:** the architecture builds a useful bias into the computation, so the model does not need to rediscover that structure from data alone.
- **Output to expect:** classification models return class scores/probabilities, reconstruction models return reconstructed inputs, and generative models return new or denoised samples.


## Visual Intuition

Run this cell before or after training. It is lightweight and framework-independent, so it explains the network idea without requiring a long training run.


In [ ]:
import numpy as np
import matplotlib.pyplot as plt
fig, axes = plt.subplots(1, 3, figsize=(12, 3.3))
fig.suptitle("Generative Adversarial Network: intuition, signal flow, and output", fontsize=13)
axes[0].axis("off")
layers = ['noise', 'generator', 'judge']
xs = np.linspace(0.1, 0.9, len(layers))
for xpos, label in zip(xs, layers):
    axes[0].scatter([xpos], [0.55], s=1200, color="#4C78A8", alpha=0.18, edgecolors="#4C78A8")
    axes[0].text(xpos, 0.55, label, ha="center", va="center", fontsize=9)
for a, b in zip(xs[:-1], xs[1:]):
    axes[0].annotate("", xy=(b - 0.045, 0.55), xytext=(a + 0.045, 0.55), arrowprops=dict(arrowstyle="->", lw=1.5))
axes[0].set_title("How data moves")
x = np.linspace(-3, 3, 160)
y = 1/(1+np.exp(-x))
axes[1].plot(x, y, color="#F58518", lw=2)
axes[1].axhline(0, color="black", lw=0.5)
axes[1].set_title("Toy behavior")
axes[1].grid(alpha=0.25)
values = np.array([.48,.52])
axes[2].bar(range(len(values)), values, color=["#54A24B", "#E45756", "#72B7B2", "#B279A2"][:len(values)])
axes[2].set_title("Typical output")
axes[2].set_xticks(range(len(values)))
axes[2].set_xticklabels(['real', 'fake'])
axes[2].grid(axis="y", alpha=0.25)
plt.tight_layout()
plt.show()


In [ ]:
import os
import random
import numpy as np

SEED = 42
random.seed(SEED)
np.random.seed(SEED)

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

tf.keras.utils.set_random_seed(SEED)
print(f"TensorFlow: {tf.__version__}")
from sklearn.datasets import load_digits

X = (load_digits().data / 16.0).astype("float32")
dataset = tf.data.Dataset.from_tensor_slices(X).shuffle(1024).batch(64, drop_remainder=True)
latent_dim = 32


In [ ]:
generator = keras.Sequential([
    layers.Input(shape=(latent_dim,)),
    layers.Dense(64),
    layers.LeakyReLU(0.2),
    layers.Dense(128),
    layers.LeakyReLU(0.2),
    layers.Dense(64, activation="sigmoid"),
])
discriminator = keras.Sequential([
    layers.Input(shape=(64,)),
    layers.Dense(128),
    layers.LeakyReLU(0.2),
    layers.Dropout(0.2),
    layers.Dense(1),
])
opt_g = keras.optimizers.AdamW(2e-4)
opt_d = keras.optimizers.AdamW(2e-4)
loss_fn = keras.losses.BinaryCrossentropy(from_logits=True)


In [ ]:
for epoch in range(20):
    for real in dataset:
        batch_size = tf.shape(real)[0]
        noise = tf.random.normal((batch_size, latent_dim))

        with tf.GradientTape() as d_tape:
            fake = generator(noise, training=True)
            d_loss = loss_fn(tf.ones((batch_size, 1)), discriminator(real, training=True))
            d_loss += loss_fn(tf.zeros((batch_size, 1)), discriminator(fake, training=True))
        opt_d.apply_gradients(zip(d_tape.gradient(d_loss, discriminator.trainable_variables), discriminator.trainable_variables))

        with tf.GradientTape() as g_tape:
            generated = generator(tf.random.normal((batch_size, latent_dim)), training=True)
            g_loss = loss_fn(tf.ones((batch_size, 1)), discriminator(generated, training=True))
        opt_g.apply_gradients(zip(g_tape.gradient(g_loss, generator.trainable_variables), generator.trainable_variables))

    print(f"epoch={epoch+1:02d} d_loss={float(d_loss):.3f} g_loss={float(g_loss):.3f}")
